In [4]:
!pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable


In [16]:
# ==========================================
# Import Required Libraries
# ==========================================

# Numerical Computing
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Set Random Seed for Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


In [17]:
# ==========================================
# Create Smart Irrigation Dataset
# ==========================================

df = pd.DataFrame({
    "soil_moisture": [
        0.08, 0.12, 0.18, 0.22, 0.28, 0.35, 0.42, 0.50, 0.58, 0.65,
        0.72, 0.80, 0.15, 0.25, 0.33, 0.45, 0.55, 0.62, 0.10, 0.20,
        0.30, 0.40, 0.52, 0.68, 0.75, 0.18, 0.27, 0.38, 0.48, 0.60
    ],

    "temperature_c": [
        36, 34, 32, 30, 28, 27, 26, 29, 31, 33,
        35, 37, 25, 24, 23, 26, 28, 30, 38, 35,
        32, 29, 27, 25, 24, 36, 31, 28, 26, 34
    ],

    "sunlight_hours": [
        10, 9, 8, 7, 6, 5, 4, 7, 8, 9,
        11, 12, 3, 4, 5, 6, 7, 8, 10, 9,
        8, 7, 6, 5, 4, 11, 9, 6, 5, 10
    ],

    "humidity": [
        30, 35, 40, 45, 50, 55, 60, 65, 70, 75,
        80, 85, 50, 55, 60, 65, 70, 75, 35, 40,
        45, 50, 60, 70, 80, 30, 45, 55, 65, 75
    ],

    "rainfall_mm": [
        0, 0, 2, 5, 10, 15, 20, 8, 5, 2,
        0, 0, 18, 22, 25, 12, 8, 5, 0, 1,
        3, 10, 18, 22, 28, 0, 2, 12, 18, 0
    ],

    "need_water": [
        1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 1, 0, 0, 0, 0, 1, 1,
        1, 0, 0, 0, 0, 1, 1, 0, 0, 0
    ]
})

# Display first five rows
df.head()

,soil_moisture,temperature_c,sunlight_hours,humidity,rainfall_mm,need_water
0,0.08,36,10,30,0,1
1,0.12,34,9,35,0,1
2,0.18,32,8,40,2,1
3,0.22,30,7,45,5,1
4,0.28,28,6,50,10,0


In [7]:
df.columns

Index(['soil_moisture', 'temperature_c', 'sunlight_hours', 'need_water'], dtype='object')

In [18]:
X = df[['soil_moisture', 'temperature_c', 'sunlight_hours', 'humidity', 'rainfall_mm']]
y = df['need_water']

In [19]:
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# Feature Scaling using MinMaxScaler
# ==========================================

scaler = MinMaxScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

# View scaled data
X_scaled.head()

,soil_moisture,temperature_c,sunlight_hours,humidity,rainfall_mm
0,0.000000,0.866667,0.777778,0.000000,0.000000
1,0.055556,0.733333,0.666667,0.090909,0.000000
2,0.138889,0.600000,0.555556,0.181818,0.071429
3,0.194444,0.466667,0.444444,0.272727,0.178571
4,0.277778,0.333333,0.333333,0.363636,0.357143


In [20]:
# ==========================================
# Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
    shuffle=True
)

print("Training Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])

# ==========================================
# Build Artificial Neural Network (ANN)
# ==========================================

model = keras.Sequential([

    # Input Layer (5 features)
    layers.Input(shape=(X_train.shape[1],)),

    # Hidden Layer 1
    layers.Dense(
        16,
        activation="relu",
        kernel_initializer="he_normal"
    ),
    layers.Dropout(0.20),

    # Hidden Layer 2
    layers.Dense(
        8,
        activation="relu"
    ),

    # Output Layer
    layers.Dense(
        1,
        activation="sigmoid"
    )
])

# Display Model Architecture
model.summary()

Training Samples : 24
Testing Samples  : 6


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ dense_3 (Dense)               │ (None, 16)            │           96 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dropout_1 (Dropout)           │ (None, 16)            │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_4 (Dense)               │ (None, 8)             │          136 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_5 (Dense)               │ (None, 1)             │            9 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
# ==========================================
# Function to Build ANN Model
# ==========================================

def build_model():

    model = keras.Sequential([

        layers.Input(shape=(X_train.shape[1],)),

        layers.Dense(
            16,
            activation="relu",
            kernel_initializer="he_normal"
        ),

        layers.Dropout(0.20),

        layers.Dense(
            8,
            activation="relu"
        ),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    optimizer = keras.optimizers.Adam(learning_rate=0.001)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [22]:
print("="*50)
print("Training using Full Batch Gradient Descent")
print("="*50)

model_full = build_model()

history_full = model_full.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=len(X_train),
    verbose=1
)

Training using Full Batch Gradient Descent
Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3158 - loss: 1.2818 - val_accuracy: 0.6000 - val_loss: 0.8331
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.3158 - loss: 1.3751 - val_accuracy: 0.6000 - val_loss: 0.8258
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.3158 - loss: 1.3840 - val_accuracy: 0.6000 - val_loss: 0.8185
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.3158 - loss: 1.3761 - val_accuracy: 0.6000 - val_loss: 0.8113
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.3158 - loss: 1.1788 - val_accuracy: 0.6000 - val_loss: 0.8042
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.3158 - loss: 1.2397 - val_accuracy: 0.6000 - val_loss: 0.7975
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.3158 - loss: 1.3611 - val_accuracy: 0.6000 - val_loss: 0.7908
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.3158 - loss: 

In [23]:
print("="*50)
print("Training using Stochastic Gradient Descent")
print("="*50)

model_sgd = build_model()

history_sgd = model_sgd.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=1,
    verbose=1
)

Training using Stochastic Gradient Descent
Epoch 1/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3158 - loss: 1.0324 - val_accuracy: 0.6000 - val_loss: 0.5484
Epoch 2/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3684 - loss: 0.8864 - val_accuracy: 0.6000 - val_loss: 0.4989
Epoch 3/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4211 - loss: 0.8368 - val_accuracy: 0.6000 - val_loss: 0.4650
Epoch 4/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3158 - loss: 0.8120 - val_accuracy: 0.8000 - val_loss: 0.4403
Epoch 5/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4737 - loss: 0.6355 - val_accuracy: 0.8000 - val_loss: 0.4276
Epoch 6/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6316 - loss: 0.6831 - val_accuracy: 1.0000 - val_loss: 0.4186
Epoch 7/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6842 - loss: 0.6512 - val_accuracy: 1.0000 - val_loss: 0.4253
Epoch 8/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.78

In [24]:
print("="*50)
print("Training using Mini-Batch Gradient Descent")
print("="*50)

model_mini = build_model()

history_mini = model_mini.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=4,
    verbose=1
)

Training using Mini-Batch Gradient Descent
Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.6842 - loss: 0.6436 - val_accuracy: 0.4000 - val_loss: 0.8617
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6842 - loss: 0.6374 - val_accuracy: 0.4000 - val_loss: 0.8450
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6842 - loss: 0.6139 - val_accuracy: 0.4000 - val_loss: 0.8275
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6842 - loss: 0.5972 - val_accuracy: 0.4000 - val_loss: 0.8111
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6842 - loss: 0.5976 - val_accuracy: 0.4000 - val_loss: 0.7973
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6842 - loss: 0.5617 - val_accuracy: 0.4000 - val_loss: 0.7813
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6842 - loss: 0.5443 - val_accuracy: 0.4000 - val_loss: 0.7661
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6842 - loss

In [25]:
print("\nFinal Validation Accuracy")

print(f"Full Batch : {history_full.history['val_accuracy'][-1]:.4f}")

print(f"SGD        : {history_sgd.history['val_accuracy'][-1]:.4f}")

print(f"Mini Batch : {history_mini.history['val_accuracy'][-1]:.4f}")


Final Validation Accuracy
Full Batch : 0.8000
SGD        : 1.0000
Mini Batch : 1.0000
